## Problem 1

### (a)

By Itô,
$$
\mathrm{d}C_t
=
\left(
\frac{\partial C}{\partial t}
+\alpha(r_t,t)\frac{\partial C}{\partial r}
+\frac{1}{2}\beta(r_t,t)^2\frac{\partial^2 C}{\partial r^2}
\right)\mathrm{d}t
+
\beta(r_t,t)\frac{\partial C}{\partial r}\,\mathrm{d}W_t .
$$

Under the risk-neutral measure, the drift of a traded price process must be $r_t C_t$. Hence
$$
\frac{\partial C}{\partial t}
+\alpha(r,t)\frac{\partial C}{\partial r}
+\frac{1}{2}\beta(r,t)^2\frac{\partial^2 C}{\partial r^2}
=
rC,
\qquad
C(r,T)=F(r).
$$

For the Vasicek model,
$$
\frac{\partial C}{\partial t}
+\kappa(\theta-r)\frac{\partial C}{\partial r}
+\frac{1}{2}\sigma^2\frac{\partial^2 C}{\partial r^2}
=
rC,
\qquad
C(r,T)=1.
$$


In [1]:
import numpy as np

In [2]:
class Vasicek:

    def __init__(self,kappa,theta,sigma):
        self.kappa=kappa
        self.theta=theta
        self.sigma=sigma

In [3]:
hw41dynamics = Vasicek(kappa=3,theta=0.05,sigma=0.03)

In [4]:
class Bond:

    def __init__(self, T):
        self.T=T


In [5]:
hw41contract = Bond(T=5)

In [6]:
class FDexplicitEngine:

    def __init__(self, rMax, rMin, deltar, deltat, useUpwind):
        self.rMax=rMax
        self.rMin=rMin
        self.deltar=deltar
        self.deltat=deltat
        self.useUpwind=useUpwind

    def price_bond_vasicek(self,contract,dynamics):
    # Returns array of all initial short rates,
    # and the corresponding array of zero-coupon
    # T-maturity bond prices

        T = contract.T
        N=round(T/self.deltat)
        if abs(N-T/self.deltat) > 1e-12:
            raise ValueError("Bad delta t")

        r=np.arange(self.rMax,self.rMin-self.deltar/2,-self.deltar)
        bondprice=np.ones(np.size(r))

        mu = dynamics.kappa*(dynamics.theta-r)
        diff = 0.5*dynamics.sigma**2*self.deltat/self.deltar**2
        drift = mu*self.deltat/self.deltar

        if self.useUpwind:
            qu=np.where(mu>=0, diff+drift, diff)
            qd=np.where(mu>=0, diff, diff-drift)
            qm=np.where(mu>=0, 1-2*diff-drift, 1-2*diff+drift)
        else:
            qu=diff+0.5*drift
            qd=diff-0.5*drift
            qm=np.full_like(r, 1-2*diff)

        for t in np.arange(N-1,-1,-1)*self.deltat:
            bondprice[1:-1]=1/(1+r[1:-1]*self.deltat)*(qd[1:-1]*bondprice[2:]+qm[1:-1]*bondprice[1:-1]+qu[1:-1]*bondprice[:-2])

            bondprice[0]=2*bondprice[1]-bondprice[2]
            bondprice[-1]=2*bondprice[-2]-bondprice[-3]

        return (r, bondprice)

In [7]:
hw41FD = FDexplicitEngine(rMax=0.35,rMin=-0.25,deltar=0.01,deltat=0.01,useUpwind=False)

In [8]:
(r, bondprice) = hw41FD.price_bond_vasicek(hw41contract,hw41dynamics)

In [9]:
np.set_printoptions(precision=4,suppress=True)
displayrows=(r<0.15+hw41FD.deltar/2) & (r>0.0-hw41FD.deltar/2)

In [10]:
print(np.stack((r, bondprice),axis=1)[displayrows])

[[ 1.5000e-01 -1.4273e+09]
 [ 1.4000e-01  1.6361e+08]
 [ 1.3000e-01  2.2294e+07]
 [ 1.2000e-01 -1.3724e+06]
 [ 1.1000e-01 -1.3361e+05]
 [ 1.0000e-01  3.2966e+03]
 [ 9.0000e-02  1.3021e+02]
 [ 8.0000e-02  7.7128e-01]
 [ 7.0000e-02  7.7385e-01]
 [ 6.0000e-02  7.7643e-01]
 [ 5.0000e-02  7.7902e-01]
 [ 4.0000e-02  7.8162e-01]
 [ 3.0000e-02  7.8423e-01]
 [ 2.0000e-02  7.8685e-01]
 [ 1.0000e-02  1.4165e+03]
 [-3.3307e-16  5.1498e+04]]


In [11]:
def vasicek_bond_exact(r0, T, dynamics):
    kappa = dynamics.kappa
    theta = dynamics.theta
    sigma = dynamics.sigma
    B = (1-np.exp(-kappa*T))/kappa
    A = np.exp((theta-sigma**2/(2*kappa**2))*(B-T)-sigma**2*B**2/(4*kappa))
    return A*np.exp(-B*r0)

hw41FD_upwind = FDexplicitEngine(rMax=0.35,rMin=-0.25,deltar=0.01,deltat=0.01,useUpwind=True)
(r_upwind, bondprice_upwind) = hw41FD_upwind.price_bond_vasicek(hw41contract,hw41dynamics)

r0 = 0.10
central_price = bondprice[np.argmin(np.abs(r-r0))]
upwind_price = bondprice_upwind[np.argmin(np.abs(r_upwind-r0))]
exact_price = vasicek_bond_exact(r0, hw41contract.T, hw41dynamics)

print("central :", central_price)
print("upwind  :", upwind_price)
print("exact   :", exact_price)

for r0 in [0.12, 0.02]:
    P = bondprice_upwind[np.argmin(np.abs(r_upwind-r0))]
    y = np.log(1/P)/hw41contract.T
    print(f"r0={r0:.2f}, bond price={P:.6f}, yield={y:.6f}")

central : 3296.5929237489718
upwind  : 0.7662252882450689
exact   : 0.766100695543474
r0=0.12, bond price=0.761144, yield=0.054587
r0=0.02, bond price=0.786916, yield=0.047927


### (d)

Taylor around $x$:
$$
f(x+h)=f(x)+hf'(x)+\frac{h^2}{2}f''(x)+O(h^3),
\qquad
f(x-h)=f(x)-hf'(x)+\frac{h^2}{2}f''(x)+O(h^3).
$$

So
$$
\frac{f(x+h)-f(x)}{h}
=
f'(x)+\frac{h}{2}f''(x)+O(h^2),
$$
hence
$$
\left|\frac{f(x+h)-f(x)}{h}-f'(x)\right|=O(h).
$$

Also
$$
f(x+h)-f(x-h)=2hf'(x)+O(h^3),
$$
so
$$
\frac{f(x+h)-f(x-h)}{2h}
=
f'(x)+O(h^2),
$$
hence
$$
\left|\frac{f(x+h)-f(x-h)}{2h}-f'(x)\right|=O(h^2).
$$

### (e)

With $\Delta r=0.01$, $\Delta t=0.01$, $r_{\max}=0.35$, $r_{\min}=-0.25$, and $r_0=0.10$:

- central difference: very inaccurate
- upwind: $0.766225$

The exact Vasicek bond price is $0.766101$, so the upwind scheme is the good one here.

### (f)

Ignoring stability and considering only consistency, the upwind explicit scheme has **less** accuracy than the central explicit scheme.

In a PDE with strong drift, the upwind scheme may allow **greater** freedom in choosing grid spacing.

### (g)

Using the good upwind grid solution:
$$
y(0,5)=\frac{\log(1/P_0)}{5}.
$$

- if $r_0=0.12$: $P_0 \approx 0.761144$, so $y \approx 0.054587$
- if $r_0=0.02$: $P_0 \approx 0.786916$, so $y \approx 0.047927$

Because In Vasicek, the short rate mean-reverts toward $\theta=0.05$. So:

- when $r_0=0.12$, the future expected short rates are pulled down toward $0.05$, so the 5-year average expected short rate is below $0.12$
- when $r_0=0.02$, the future expected short rates are pulled up toward $0.05$, so the 5-year average expected short rate is above $0.02$


## Problem 2

### (a)

For
$$
\mathrm{d}X_t=\sigma X_t^{1+\alpha}\,\mathrm{d}W_t,
$$
the option price $C(X,t)$ satisfies
$$
\frac{\partial C}{\partial t}
+\frac{1}{2}\sigma^2 X^{2+2\alpha}\frac{\partial^2 C}{\partial X^2}
=
rC,
\qquad
C(X,T)=(K-X)^+.
$$

Equivalently,
$$
\frac{\partial C}{\partial t}
+\frac{1}{2}\sigma^2 X^{2+2\alpha}\frac{\partial^2 C}{\partial X^2}
-rC=0.
$$


In [12]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

In [13]:
class CEV:

    def __init__(self,volcoeff,alpha,rGrow,r,X0):
        self.volcoeff = volcoeff
        self.alpha = alpha
        self.rGrow = rGrow
        self.r = r
        self.X0 = X0


In [14]:
hw42dynamics = CEV(volcoeff=3, alpha=-0.5, rGrow=0, r=0.05, X0=100)

In [15]:
class Put:

    def __init__(self,T,K):
        self.T = T;
        self.K = K;

In [16]:
hw42contract=Put(T=0.25,K=100)

In [17]:
class FD_CrankNicolson_Engine:

    def __init__(self,XMax,XMin,deltaX,deltat):
        self.XMax=XMax
        self.XMin=XMin
        self.deltaX=deltaX
        self.deltat=deltat

    def TicksAndMatricesCEV(self,T,dynamics):

        alpha, r, rGrow, volcoeff = dynamics.alpha, dynamics.r, dynamics.rGrow, dynamics.volcoeff

        N=round(T/self.deltat)
        if abs(N-T/self.deltat)>1e-12:
            raise ValueError('Bad time step')
        numX=round((self.XMax-self.XMin)/self.deltaX)+1
        if abs(numX-(self.XMax-self.XMin)/self.deltaX-1)>1e-12:
            raise ValueError('Bad time step')
        X=np.linspace(self.XMax,self.XMin,numX)
        tTicks = np.arange(N-1,-1,-1)*self.deltat

        ratio1 = self.deltat/self.deltaX
        ratio2 = self.deltat/self.deltaX**2

        f = 0.5*volcoeff**2*X**(2+2*alpha)
        g = np.zeros_like(X)
        h = -r

        F = 0.5*ratio2*f + 0.25*ratio1*g
        G =     ratio2*f - 0.50*self.deltat*h
        H = 0.5*ratio2*f - 0.25*ratio1*g

        RHSmatrix = diags([H[:-1], 1-G, F[1:]], [1,0,-1], shape=(numX,numX), format="csr")
        LHSmatrix = diags([-H[:-1], 1+G, -F[1:]], [1,0,-1], shape=(numX,numX), format="csr")

        return(X, tTicks, LHSmatrix, RHSmatrix, H[-1], F[0])


    def price_put_CEV(self,contract,dynamics):

        X, tTicks, LHSmatrix, RHSmatrix, bottomH, topF = self.TicksAndMatricesCEV(contract.T,dynamics)

        putprice=np.maximum(contract.K-X,0)
        X_lowboundary=self.XMin-self.deltaX

        for t in tTicks:

            rhs = RHSmatrix @ putprice
            rhs[-1]=rhs[-1]+2*bottomH*(contract.K-X_lowboundary)

            putprice = spsolve(LHSmatrix, rhs)

            putprice = np.maximum(putprice, contract.K-X)

        return(X, putprice)

In [18]:
hw42FD = FD_CrankNicolson_Engine(XMax=200,XMin=50,deltaX=0.1,deltat=0.0005)

In [19]:
(X0_all, putprice) = hw42FD.price_put_CEV(hw42contract,hw42dynamics)

In [20]:
# pricer_put_CEV_CrankNicolson gives us option prices for ALL X0 from XMin to XMax
# But let's display only for a few X0 near 100:

displayStart = hw42dynamics.X0-hw42FD.deltaX*1.5
displayEnd   = hw42dynamics.X0+hw42FD.deltaX*1.5
displayrows  = (X0_all>displayStart) & (X0_all<displayEnd)
np.set_printoptions(precision=4, suppress=True)
print(np.stack((X0_all, putprice),axis=1)[displayrows])

[[100.1      5.8704]
 [100.       5.9183]
 [ 99.9      5.9665]]


In [21]:
i0 = np.argmin(np.abs(X0_all-hw42dynamics.X0))
delta = (putprice[i0-1]-putprice[i0+1])/(X0_all[i0-1]-X0_all[i0+1])
gamma = (putprice[i0-1]-2*putprice[i0]+putprice[i0+1])/(hw42FD.deltaX**2)

print("CEV American put price at X0=100:", putprice[i0])
print("delta:", delta)
print("gamma:", gamma)

hw42dynamics_bs = CEV(volcoeff=0.30, alpha=0.0, rGrow=0, r=0.05, X0=100)
(X0_all_bs, putprice_bs) = hw42FD.price_put_CEV(hw42contract, hw42dynamics_bs)
i0_bs = np.argmin(np.abs(X0_all_bs-hw42dynamics_bs.X0))
print("Black-Scholes American put price at X0=100:", putprice_bs[i0_bs])

CEV American put price at X0=100: 5.918298433328996
delta: -0.4806400882508675
gamma: 0.026399534382726127


Black-Scholes American put price at X0=100: 5.916922913148806


### (c)

From the grid at $X_0=100$,

$$
\Delta \approx \frac{C(100.1,0)-C(99.9,0)}{0.2} \approx -0.480640,
$$
$$
\Gamma \approx \frac{C(100.1,0)-2C(100,0)+C(99.9,0)}{(0.1)^2} \approx 0.026400.
$$

### (d)

Using the same Crank-Nicolson engine with Black-Scholes dynamics
($\alpha=0$, $\sigma=0.30$, $r=0.05$, $X_0=100$) gives

$$
C(100,0)\approx 5.916923.
$$

### (e)

Under the CEV dynamics in part (b),
$$
\text{local percentage volatility}=\sigma X^\alpha = 3X^{-1/2},
$$
which is larger when $X$ is lower. So downside states carry higher future volatility than upside states. That produces a **downward-sloping implied volatility skew**: low-strike puts have higher implied vol than high-strike options.

Under Black-Scholes in part (d), percentage volatility is constant at $30\%$, so the European implied volatility curve is **flat** across strike.
